# Workshop 2: RAG + LangGraph Agent with MCP Tools

Build a **Boilermaker TA** — an autonomous agent that:
1. Retrieves facts from a course knowledge base using **FAISS + embeddings (RAG)**  
2. Queries an academic calendar with the same retrieval approach  
3. Writes student announcement notifications  
4. Orchestrates everything through a **LangGraph** stateful agent graph  
5. Exposes tools via the **MCP** (Model Context Protocol) over HTTP

**What you'll do:**
- Part A: RAG from scratch — embed, index, retrieve, generate  
- Part B: MCP server — expose tools over HTTP with FastMCP  
- Part C: LangGraph agent — connect the LLM to MCP tools in a loop

## 0. Install dependencies

Run once, then restart the kernel.

In [ ]:
# !pip install torch transformers accelerate sentence-transformers faiss-cpu \
#              langchain langchain-core langgraph langchain-mcp-adapters fastmcp python-dotenv

---
# Part A: Retrieval-Augmented Generation (RAG)

RAG = embed documents → store in a vector index → at query time, retrieve the most relevant chunks → pass them as context to the LLM.

## A1. Imports and data loading

In [ ]:
import os
import json
from pathlib import Path
from typing import List

import faiss
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

BASE_DIR = Path(".").resolve()
DATA_DIR = BASE_DIR / "boilermaker_ta_data"

print("Data directory:", DATA_DIR)
print("Files:", list(DATA_DIR.iterdir()) if DATA_DIR.exists() else "NOT FOUND")

## A2. Load the knowledge base corpus

In [ ]:
def _load_corpus():
    path = DATA_DIR / "knowledge_base.json"
    if not path.exists():
        # Fallback so the notebook runs even without the data file
        return [
            {
                "title": "Fallback: RAG Intro",
                "text": "This is a fallback document. Add boilermaker_ta_data/knowledge_base.json for workshop data.",
            }
        ]
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


CORPUS = _load_corpus()
print(f"Loaded {len(CORPUS)} documents")
print("First doc:", CORPUS[0]["title"])

## A3. Build the FAISS retriever

Steps:
1. **Embed** each document with `all-MiniLM-L6-v2` (384-dim vectors)  
2. **L2-normalize** vectors so inner product = cosine similarity  
3. **Index** in a FAISS `IndexFlatIP` (brute-force inner product search)  

In production, save/load the index from disk to avoid re-embedding on every startup.

In [ ]:
def build_retriever(model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
    embedder  = SentenceTransformer(model_name)
    texts     = [doc["text"] for doc in CORPUS]
    metadatas = [{"title": doc.get("title") or f"doc{i}"} for i, doc in enumerate(CORPUS)]

    embeddings = embedder.encode(texts, convert_to_numpy=True, show_progress_bar=True)
    if embeddings.ndim == 1:
        embeddings = embeddings.reshape(1, -1)

    faiss.normalize_L2(embeddings)          # normalize so dot product = cosine sim
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)

    return {"embedder": embedder, "index": index, "texts": texts, "metadatas": metadatas}


retriever = build_retriever()
print(f"FAISS index has {retriever['index'].ntotal} vectors of dim {retriever['index'].d}")

## A4. Retrieve documents for a query

In [ ]:
def retrieve_docs(retriever, query: str, k: int = 3):
    q_emb = retriever["embedder"].encode(query, convert_to_numpy=True)
    if q_emb.ndim == 1:
        q_emb = q_emb.reshape(1, -1)
    faiss.normalize_L2(q_emb)
    scores, ids = retriever["index"].search(q_emb, min(k, len(retriever["texts"])))
    return [
        {
            "text":     retriever["texts"][idx],
            "metadata": retriever["metadatas"][idx],
            "score":    float(scores[0][i]),
        }
        for i, idx in enumerate(ids[0])
    ]


# Test retrieval
query = "When are the office hours for this course?"
docs  = retrieve_docs(retriever, query, k=3)
for d in docs:
    print(f"[{d['score']:.3f}] {d['metadata']['title']}")
    print("  ", d["text"][:120], "...\n")

## A5. Build a RAG prompt and generate an answer

The retrieved context is inserted into the prompt so the LLM can cite it directly.

In [ ]:
def make_prompt(query: str, context: str) -> str:
    return (
        "You are a RAG assistant. Use the retrieved context to answer the user query.\n\n"
        "Context:\n" + context +
        "\n\nQuestion:\n" + query +
        "\n\nAnswer in a concise, factual way."
    )


context = "\n\n".join([d["text"] for d in docs])
prompt  = make_prompt(query, context)
print(prompt)

## A6. Load the local LLM

`LOCAL_MODEL` env var lets you swap models without changing code.  
`device_map="auto"` places layers on GPU if available, CPU otherwise.  
`bfloat16` is only requested when CUDA is available — avoids issues on CPU-only machines.

In [ ]:
def get_llm():
    local_model_name = os.environ.get("LOCAL_MODEL", "tiiuae/falcon-7b-instruct")
    tokenizer    = AutoTokenizer.from_pretrained(local_model_name)
    model_kwargs = {"device_map": "auto"}
    if torch.cuda.is_available():
        model_kwargs["torch_dtype"] = torch.bfloat16   # bfloat16 only safe with CUDA

    local_model = AutoModelForCausalLM.from_pretrained(local_model_name, **model_kwargs)
    return pipeline("text-generation", model=local_model, tokenizer=tokenizer, max_new_tokens=256)


print("Loading LLM... (may take a minute)")
llm = get_llm()
print("LLM ready.")

## A7. Run RAG queries end-to-end

In [ ]:
def answer_queries(queries: List[str], llm, retriever):
    for query in queries:
        print(f"\n=== QUERY: {query}\n")
        docs    = retrieve_docs(retriever, query, k=3)
        context = "\n\n".join([d["text"] for d in docs])
        prompt  = make_prompt(query, context)
        output  = llm(prompt, return_full_text=False)[0]["generated_text"]
        print("Answer:\n", output)
        print("Sources:\n", [d["metadata"] for d in docs])


queries = [
    "When are the office hours for this course?",
    "When are assignments due and what is the usual deadline?",
    "When are the exam weeks scheduled?",
    "What tools and resources does the course recommend for projects?",
    "When is the final project presentation scheduled?",
]

answer_queries(queries, llm, retriever)

---
# Part B: MCP Server — Exposing Tools over HTTP

**MCP (Model Context Protocol)** is a standard for exposing tools that LLMs can call.  
We run the MCP server in a background thread so the full demo runs inside this notebook.

**Transport options:**

| Transport | How it works | Best for |
|---|---|---|
| `http` (Streamable HTTP) | Agent connects via HTTP; server runs independently | Production, multi-client, debuggable |
| `stdio` | Client spawns server as a subprocess | CLI tools, local single-client use |

## B1. Imports for the MCP server

In [ ]:
import re
import threading
import time

from fastmcp import FastMCP

## B2. SimpleRetriever — shared retrieval helper for MCP tools

In [ ]:
class SimpleRetriever:
    def __init__(self, texts, metadatas, model_name="sentence-transformers/all-MiniLM-L6-v2"):
        self.embedder  = SentenceTransformer(model_name)
        self.texts     = texts
        self.metadatas = metadatas
        self.index     = self._build_index(texts)

    def _build_index(self, texts):
        embeddings = self.embedder.encode(texts, convert_to_numpy=True, show_progress_bar=False)
        if embeddings.ndim == 1:
            embeddings = embeddings.reshape(1, -1)
        faiss.normalize_L2(embeddings)
        index = faiss.IndexFlatIP(embeddings.shape[1])
        index.add(embeddings)
        return index

    def similarity_search(self, query: str, k: int = 3):
        q_emb = self.embedder.encode(query, convert_to_numpy=True)
        if q_emb.ndim == 1:
            q_emb = q_emb.reshape(1, -1)
        faiss.normalize_L2(q_emb)
        scores, ids = self.index.search(q_emb, min(k, len(self.texts)))
        return [
            type("Doc", (), {"page_content": self.texts[idx], "metadata": self.metadatas[idx]})
            for idx in ids[0]
        ]

## B3. Retriever builder helpers

In [ ]:
ANNOUNCEMENTS_FILE = BASE_DIR / "announcements.txt"


def _load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def _build_retriever():
    """Build a FAISS retriever from the knowledge base.
    For production, cache the vectorstore to avoid rebuilding on every call.
    """
    kb        = _load_json(DATA_DIR / "knowledge_base.json")
    texts     = [doc["text"] for doc in kb]
    metadatas = [{"title": doc.get("title")} for doc in kb]
    return SimpleRetriever(texts, metadatas)


def _build_calendar_retriever():
    """Build a FAISS retriever from academic calendar events.
    Calendar events are flattened to 'date title notes' strings for embedding.
    """
    calendar = _load_json(DATA_DIR / "purdue_calendar.json")
    events   = calendar.get("events", [])
    if not events:
        return None
    texts     = [f"{e['date']} {e['title']} {e['notes']}" for e in events]
    metadatas = [{"date": e.get("date"), "title": e.get("title")} for e in events]
    return SimpleRetriever(texts, metadatas)

## B4. Define the MCP app and its tools

Each `@app.tool` function becomes a tool the LLM agent can call by name.

In [ ]:
app = FastMCP(
    name="boilermaker-ta",
    instructions=(
        "Provides tools for finding course facts, querying the academic calendar, "
        "and writing student notification announcements."
    ),
)


@app.tool
def search_knowledge_base(query: str) -> str:
    """Search a small local knowledge base of course and schedule facts."""
    kb          = _load_json(DATA_DIR / "knowledge_base.json")
    query_terms = re.findall(r"\w+", query.lower())
    scored = []
    for doc in kb:
        text  = doc["text"].lower()
        score = sum(text.count(term) for term in query_terms)
        if score > 0:
            scored.append((score, doc))
    if not scored:
        return "No relevant knowledge found. Try a different question."
    scored.sort(key=lambda item: item[0], reverse=True)
    return "\n\n".join(f"{doc['title']}:\n{doc['text']}" for _, doc in scored[:2])


@app.tool
def retrieve_docs(query: str, k: int = 3) -> str:
    """Retrieve top-k documents from the knowledge base using embeddings + FAISS."""
    try:
        r = _build_retriever()
    except Exception as e:
        return f"Failed to build retriever: {e}"
    docs  = r.similarity_search(query, k=k)
    lines = []
    for d in docs:
        title   = d.metadata.get("title") if d.metadata else "(no title)"
        excerpt = (d.page_content[:400] + "...") if len(d.page_content) > 400 else d.page_content
        lines.append(f"{title}: {excerpt}")
    return "\n\n".join(lines)


@app.tool
def get_academic_calendar(query: str = "next 30 days") -> str:
    """Retrieve calendar events using semantic search (embedding + FAISS)."""
    try:
        r = _build_calendar_retriever()
    except Exception as e:
        return f"Failed to build calendar retriever: {e}"
    if r is None:
        return "Academic calendar is empty."
    docs  = r.similarity_search(query, k=5)
    lines = [
        f"{d.metadata.get('date')} - {d.metadata.get('title')}"
        for d in docs
    ]
    return "Academic calendar events:\n" + "\n".join(lines)


@app.tool
def create_notification(subject: str, body: str) -> str:
    """Write a notification to announcements.txt and return its location."""
    ANNOUNCEMENTS_FILE.parent.mkdir(parents=True, exist_ok=True)
    entry = f"Subject: {subject}\n{body}\n---\n"
    with open(ANNOUNCEMENTS_FILE, "a", encoding="utf-8") as f:
        f.write(entry)
    return f"Notification written to {ANNOUNCEMENTS_FILE}."


print("MCP app defined with tools:", [t.name for t in app._tool_manager.list_tools()])

## B5. Start the MCP server in a background thread

Normally you'd run the server in a separate terminal (`python step2_boiler_ta_server.py`).  
Here we use a `threading.Thread` so the full demo stays in one notebook.

In [ ]:
MCP_HOST = os.environ.get("MCP_HOST", "127.0.0.1")
MCP_PORT = int(os.environ.get("MCP_PORT", "8001"))


def _start_server():
    # Run the FastMCP server with HTTP transport (Streamable HTTP)
    # This makes it accessible at http://<host>:<port>/mcp
    app.run(transport="http", host=MCP_HOST, port=MCP_PORT)


server_thread = threading.Thread(target=_start_server, daemon=True)
server_thread.start()

# Give the server a moment to bind the port before the agent tries to connect
time.sleep(2)
print(f"MCP server running at http://{MCP_HOST}:{MCP_PORT}/mcp")

---
# Part C: LangGraph Agent

The agent is a **stateful graph** with two nodes:
- `agent` — the LLM, decides whether to call a tool or return a final answer  
- `tools` — a `ToolNode` that executes whatever tool the LLM requested  

The `should_continue` router inspects the last message: if it has `tool_calls`, go to `tools`; otherwise end.

## C1. Imports for the agent

In [ ]:
import asyncio
import logging
from typing import Annotated
from typing_extensions import TypedDict

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

## C2. Configure the LLM backend

`init_chat_model()` accepts a `"provider:model"` string.  
Set `LLM_MODEL` and optionally `LLM_BASE_URL` in your `.env` file.

| `LLM_MODEL` value | Provider | Notes |
|---|---|---|
| `ollama:llama3.2` | Ollama | Default; local, free |
| `openai:gpt-4o` | OpenAI | Needs `OPENAI_API_KEY` |
| `openai:llama3.2` + `LLM_BASE_URL=http://localhost:11434/v1` | Open WebUI / vLLM | OpenAI-compatible endpoint |
| `openai:model-id` + `LLM_BASE_URL=http://localhost:1234/v1` | LM Studio | Local OpenAI-compatible |
| `openai:model-id` + `LLM_BASE_URL=https://genai.rcac.purdue.edu/api` | Purdue GenAI | Purdue campus endpoint |

In [ ]:
LLM_MODEL      = os.getenv("LLM_MODEL",    "ollama:llama3.2")
LLM_BASE_URL   = os.getenv("LLM_BASE_URL", None)
MCP_SERVER_URL = os.getenv("MCP_SERVER_URL", f"http://{MCP_HOST}:{MCP_PORT}/mcp")

print("LLM model:   ", LLM_MODEL)
print("MCP server:  ", MCP_SERVER_URL)


def create_llm():
    """
    Initialize the LLM from environment configuration.
    init_chat_model() accepts a 'provider:model' string and optional kwargs.
    base_url lets you point at custom endpoints (Open WebUI, vLLM, LM Studio, Purdue GenAI).
    """
    kwargs = {"temperature": 0}
    if LLM_BASE_URL:
        kwargs["base_url"] = LLM_BASE_URL
    return init_chat_model(LLM_MODEL, **kwargs)

## C3. AgentState — the shared message history

In [ ]:
class AgentState(TypedDict):
    # add_messages is a reducer: new messages are appended, not replaced
    messages: Annotated[list, add_messages]

## C4. Build the LangGraph agent graph

```
START → agent ──┬─(tool_calls?)→ tools → agent (loop)
                └─(no tool calls)→ END
```

In [ ]:
def build_graph(tools):
    llm            = create_llm()
    llm_with_tools = llm.bind_tools(tools)   # tell the LLM what tools it can call

    def agent_node(state: AgentState) -> dict:
        response = llm_with_tools.invoke(state["messages"])
        return {"messages": [response]}

    def should_continue(state: AgentState) -> str:
        # Router: if the last message has tool_calls, execute them; otherwise finish
        last_message = state["messages"][-1]
        if last_message.tool_calls:
            return "tools"
        return END

    tool_node = ToolNode(tools)

    graph = StateGraph(AgentState)
    graph.add_node("agent", agent_node)
    graph.add_node("tools", tool_node)
    graph.add_edge(START, "agent")
    graph.add_conditional_edges("agent", should_continue, {"tools": "tools", END: END})
    graph.add_edge("tools", "agent")   # after tool execution, go back to agent
    return graph.compile()

## C5. System prompt for the Boilermaker TA

In [ ]:
SYSTEM_PROMPT = """
You are the Boilermaker Autonomous TA for a Purdue course.
Your job is to help students and instructors manage course planning using the tools available.

Tools:
- search_knowledge_base(query): search the course knowledge base for syllabus details, policies, and study guidance.
- get_academic_calendar(query): read academic calendar events.
- create_notification(subject, body): write a formatted announcement to the announcements file.

Always use the tools when you need facts from the knowledge base or calendar.
Use create_notification only when you have a final announcement ready to write.
"""

## C6. Connect to the MCP server and run the agent

In [ ]:
DEFAULT_QUESTION = (
    "A professor wants to schedule a CS course review session that does not conflict "
    "with holidays or exams. Summarize the plan and write a notification announcement for students."
)


async def run_agent(question: str = DEFAULT_QUESTION):
    log.info("Connecting to MCP server at %s", MCP_SERVER_URL)

    try:
        client = MultiServerMCPClient({
            "boiler_ta": {
                "url": MCP_SERVER_URL,
                "transport": "streamable_http",
            },
        })
        tools = await client.get_tools()
    except Exception as e:
        log.error("Could not connect to MCP server: %s", e)
        return

    log.info("Loaded %d tools: %s", len(tools), [t.name for t in tools])

    agent  = build_graph(tools)
    result = await agent.ainvoke({
        "messages": [SystemMessage(content=SYSTEM_PROMPT), HumanMessage(content=question)],
    })

    for msg in result["messages"]:
        if isinstance(msg, AIMessage) and msg.content:
            print("\nAgent reply:\n", msg.content)


# Run the async agent inside the notebook
await run_agent()

## C7. Try a custom question

In [ ]:
await run_agent("What assignment deadlines are coming up in the next two weeks? Summarize them for students.")

## C8. Check the announcements file

In [ ]:
if ANNOUNCEMENTS_FILE.exists():
    print(ANNOUNCEMENTS_FILE.read_text())
else:
    print("No announcements written yet.")

---
## Extension ideas

- Swap `LLM_MODEL` in `.env` to use a different backend (OpenAI, Purdue GenAI, vLLM)  
- Add a new `@app.tool` (e.g. `get_grades`, `send_email`) and re-run the agent  
- Change `MCP_PORT` and run the server standalone (`python step2_boiler_ta_server.py`) in a terminal  
- Replace the FAISS index with a persistent vector database (ChromaDB, Qdrant)  
- Connect the LoRA-fine-tuned model from Workshop 1 as the LLM backend